In [81]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# Constants
IMG_SIZE = 128
CASIA_PATH = "CASIA2"
DATASET_PATH = "Dataset"

In [82]:
# Paths for real & fake images
casia_real_path = os.path.join(CASIA_PATH, "Au")
casia_fake_path = os.path.join(CASIA_PATH, "Tp")

deepfake_train_path = os.path.join(DATASET_PATH, "Train")
deepfake_val_path = os.path.join(DATASET_PATH, "Validation")
deepfake_test_path = os.path.join(DATASET_PATH, "Test")


In [83]:
import os
import cv2
import numpy as np

# Constants
IMG_SIZE = 128

def load_images(folder, label):
    images = []
    labels = []
    for file in os.listdir(folder):
        img_path = os.path.join(folder, file)

        # Check if the file is an image
        try:
            img = cv2.imread(img_path)
            if img is None:
                print(f"Skipping {file}: Not a valid image!")
                continue  # Skip non-image files
            
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0  # Resize & Normalize
            images.append(img)
            labels.append(label)
        except Exception as e:
            print(f"Error processing {file}: {e}")
            continue  # Skip problematic files
        
    return np.array(images), np.array(labels)



In [84]:
def load_dataset_in_batches(folder, label, max_samples=5000):
    images = []
    labels = []
    count = 0

    for file in os.listdir(folder):
        if count >= max_samples:  # Stop loading after max_samples
            break
        img_path = os.path.join(folder, file)

        try:
            img = cv2.imread(img_path)
            if img is None:
                print(f"Skipping {file}: Not a valid image!")
                continue  # Skip non-image files
            
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0  # Resize & Normalize
            images.append(img)
            labels.append(label)
            count += 1
        except Exception as e:
            print(f"Error processing {file}: {e}")
            continue  # Skip problematic files

    return np.array(images), np.array(labels)

# Load CASIA2 Dataset (with batch limit)
casia_real, casia_real_labels = load_dataset_in_batches(casia_real_path, 0, max_samples=5000)
casia_fake, casia_fake_labels = load_dataset_in_batches(casia_fake_path, 1, max_samples=5000)


Skipping Thumbs.db: Not a valid image!
Skipping Thumbs.db: Not a valid image!
Skipping _list.txt: Not a valid image!


In [85]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 128
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(rescale=1.0/255)
val_datagen = ImageDataGenerator(rescale=1.0/255)
test_datagen = ImageDataGenerator(rescale=1.0/255)

train_generator = train_datagen.flow_from_directory(
    "Dataset/Train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    "Dataset/Validation",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    "Dataset/Test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

print("Data Generators Ready!")


Found 140002 images belonging to 2 classes.
Found 39428 images belonging to 2 classes.
Found 10905 images belonging to 2 classes.
Data Generators Ready!


In [86]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_cnn():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(128, activation='relu'),  # Learnable feature extraction
        Dense(64, activation='relu'),   # Stepwise reduction
        Dense(8, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')  # Binary Classification
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_cnn()
cnn_model.summary()




Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_22 (Conv2D)              │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_22 (MaxPooling2D) │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_23 (MaxPooling2D) │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_11 (Flatten)            │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 128)            │     7,372,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 8)              │           520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,401,105 (28.23 MB)

 Trainable params: 7,401,105 (28.23 MB)

 Non-trainable params: 0 (0.00 B)

In [90]:
import pennylane as qml
from pennylane import numpy as np
import tensorflow as tf

# Define Quantum Device
# n_qubits = 8
# dev = qml.device("default.qubit", wires=n_qubits)
n_qubits = 16  # Increase this as needed
dev = qml.device("default.qubit", wires=n_qubits)


@qml.qnode(dev)
def quantum_circuit(inputs):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights=np.random.random((3, n_qubits)), wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Custom Quantum Layer
class QuantumLayer(tf.keras.layers.Layer):
    def __init__(self, n_qubits=16):
        super(QuantumLayer, self).__init__()
        self.n_qubits = n_qubits

    def call(self, inputs):
        # Convert TensorFlow tensor to NumPy
        inputs = tf.cast(inputs, dtype=tf.float32)
        q_out = tf.map_fn(lambda x: tf.convert_to_tensor(quantum_circuit(x), dtype=tf.float32), inputs)
        
        # Ensure output is of shape (batch_size, n_qubits)
        return tf.reshape(q_out, (-1, self.n_qubits))
    
    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.n_qubits)  # Output shape is (batch, n_qubits)


In [91]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_hybrid_model():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(256, activation='relu'),  # Learnable feature extraction
        Dense(128, activation='relu'),   # Stepwise reduction
        Dense(16, activation='relu'),    # Final feature selection before quantum
        QuantumLayer(n_qubits=16),       # Send 8 learned features to quantum
        Dense(1, activation='sigmoid')
        
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

hybrid_model = build_hybrid_model()
hybrid_model.summary()


Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_28 (Conv2D)              │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_28 (MaxPooling2D) │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_29 (Conv2D)              │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_29 (MaxPooling2D) │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_14 (Flatten)            │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 256)            │    14,745,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_48 (Dense)                │ (None, 16)             │         2,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ quantum_layer_9 (QuantumLayer)  │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_49 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,800,225 (56.46 MB)

 Trainable params: 14,800,225 (56.46 MB)

 Non-trainable params: 0 (0.00 B)

In [92]:
# Train the Hybrid Model with generator
hybrid_model.fit(train_generator, validation_data=val_generator, epochs=5)

Epoch 1/5


/Users/omotolakehinde-osems/Documents/imageforgeryproject/myenv/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/Users/omotolakehinde-osems/Documents/imageforgeryproject/myenv/lib/python3.12/site-packages/pennylane/math/interface_utils.py:127: UserWarning: Contains tensors of types {'autograd', 'tensorflow'}; dispatch will prioritize TensorFlow, PyTorch, and Jax over Autograd. Consider replacing Autograd with vanilla NumPy.
  warnings.warn(


4376/4376 ━━━━━━━━━━━━━━━━━━━━ 20620s 5s/step - accuracy: 0.6998 - loss: 0.6251 - val_accuracy: 0.8172 - val_loss: 0.5168
Epoch 2/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 23600s 5s/step - accuracy: 0.8907 - loss: 0.3585 - val_accuracy: 0.8529 - val_loss: 0.4015
Epoch 3/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 35628s 8s/step - accuracy: 0.9336 - loss: 0.2149 - val_accuracy: 0.8732 - val_loss: 0.3504
Epoch 4/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 36856s 8s/step - accuracy: 0.9554 - loss: 0.1405 - val_accuracy: 0.8654 - val_loss: 0.3826
Epoch 5/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 51391s 12s/step - accuracy: 0.9701 - loss: 0.0965 - val_accuracy: 0.8771 - val_loss: 0.3880


In [74]:
# Train the CNN model with generator
cnn_model.fit(train_generator, validation_data=val_generator, epochs=5)

# Train the Hybrid Model with generator
hybrid_model.fit(train_generator, validation_data=val_generator, epochs=5)

# Evaluate on the test set
test_loss, test_acc = hybrid_model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc * 100:.2f}%")


Epoch 1/5


/Users/omotolakehinde-osems/Documents/imageforgeryproject/myenv/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


4376/4376 ━━━━━━━━━━━━━━━━━━━━ 341s 78ms/step - accuracy: 0.5021 - loss: 0.6940 - val_accuracy: 0.4981 - val_loss: 0.6932
Epoch 2/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 339s 77ms/step - accuracy: 0.5024 - loss: 0.6931 - val_accuracy: 0.4981 - val_loss: 0.6932
Epoch 3/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 337s 77ms/step - accuracy: 0.5018 - loss: 0.6932 - val_accuracy: 0.5019 - val_loss: 0.6931
Epoch 4/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 339s 77ms/step - accuracy: 0.5018 - loss: 0.6932 - val_accuracy: 0.4981 - val_loss: 0.6932
Epoch 5/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 1186s 271ms/step - accuracy: 0.4997 - loss: 0.6932 - val_accuracy: 0.5019 - val_loss: 0.6931
Epoch 1/5


/Users/omotolakehinde-osems/Documents/imageforgeryproject/myenv/lib/python3.12/site-packages/pennylane/math/interface_utils.py:127: UserWarning: Contains tensors of types {'autograd', 'tensorflow'}; dispatch will prioritize TensorFlow, PyTorch, and Jax over Autograd. Consider replacing Autograd with vanilla NumPy.
  warnings.warn(


4376/4376 ━━━━━━━━━━━━━━━━━━━━ 533s 121ms/step - accuracy: 0.7738 - loss: 0.4490 - val_accuracy: 0.8448 - val_loss: 0.3673
Epoch 2/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 536s 123ms/step - accuracy: 0.9336 - loss: 0.1640 - val_accuracy: 0.8705 - val_loss: 0.3055
Epoch 3/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 586s 134ms/step - accuracy: 0.9607 - loss: 0.1026 - val_accuracy: 0.8829 - val_loss: 0.2700
Epoch 4/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 538s 123ms/step - accuracy: 0.9754 - loss: 0.0660 - val_accuracy: 0.8964 - val_loss: 0.2597
Epoch 5/5
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 534s 122ms/step - accuracy: 0.9833 - loss: 0.0458 - val_accuracy: 0.9055 - val_loss: 0.2430
341/341 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - accuracy: 0.8316 - loss: 0.3821
Test Accuracy: 83.22%


In [93]:
# Evaluate CNN Model
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(test_generator)
print(f"CNN Model Test Accuracy: {cnn_test_acc * 100:.2f}%")

# Evaluate Hybrid Model
hybrid_test_loss, hybrid_test_acc = hybrid_model.evaluate(test_generator)
print(f"Hybrid Model Test Accuracy: {hybrid_test_acc * 100:.2f}%")


341/341 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.4914 - loss: 0.6942
CNN Model Test Accuracy: 49.13%
341/341 ━━━━━━━━━━━━━━━━━━━━ 598s 2s/step - accuracy: 0.8426 - loss: 0.4605
Hybrid Model Test Accuracy: 84.20%


In [76]:
import numpy as np

# Get a batch of test images and labels
test_images, test_labels = next(iter(test_generator))

# Get Predictions
cnn_preds = cnn_model.predict(test_images)
hybrid_preds = hybrid_model.predict(test_images)

# Convert to binary predictions (0 or 1)
cnn_preds_binary = np.round(cnn_preds)
hybrid_preds_binary = np.round(hybrid_preds)

# Print sample results
print("Actual Labels: ", test_labels[:10].flatten())
print("CNN Predictions: ", cnn_preds_binary[:10].flatten())
print("Hybrid Predictions: ", hybrid_preds_binary[:10].flatten())


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step
Actual Labels:  [0. 0. 0. 0. 1. 0. 1. 1. 0. 1.]
CNN Predictions:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Hybrid Predictions:  [0. 0. 1. 0. 1. 0. 0. 1. 0. 0.]


In [77]:
from sklearn.metrics import classification_report

# Get all true labels and predictions for the test set
true_labels = test_generator.classes  # True labels
cnn_predictions = np.round(cnn_model.predict(test_generator))  # CNN model predictions
hybrid_predictions = np.round(hybrid_model.predict(test_generator))  # Hybrid model predictions

# Print classification reports
print("CNN Model Performance:\n", classification_report(true_labels, cnn_predictions))
print("Hybrid Model Performance:\n", classification_report(true_labels, hybrid_predictions))


341/341 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step
341/341 ━━━━━━━━━━━━━━━━━━━━ 12s 35ms/step
CNN Model Performance:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00      5492
           1       0.50      1.00      0.66      5413

    accuracy                           0.50     10905
   macro avg       0.25      0.50      0.33     10905
weighted avg       0.25      0.50      0.33     10905

Hybrid Model Performance:
               precision    recall  f1-score   support

           0       0.50      0.44      0.47      5492
           1       0.49      0.55      0.52      5413

    accuracy                           0.50     10905
   macro avg       0.50      0.50      0.50     10905
weighted avg       0.50      0.50      0.50     10905



/Users/omotolakehinde-osems/Documents/imageforgeryproject/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/omotolakehinde-osems/Documents/imageforgeryproject/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/omotolakehinde-osems/Documents/imageforgeryproject/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division`

In [94]:
import numpy as np
import collections

# Check distribution of actual labels
label_counts = collections.Counter(test_generator.classes)
print(label_counts)  # Shows how many of each class are in the test set


Counter({np.int32(0): 5492, np.int32(1): 5413})


In [97]:


def predict_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return "Error: Invalid image!"
    
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0  # Resize & Normalize
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    
    prediction = cnn_model.predict(img)
    return "Fake" if prediction > 0.5 else "Real"

print(predict_image("Dataset/Test/Real/real_9.jpg"))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
Fake


In [96]:


def predict_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return "Error: Invalid image!"
    
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0  # Resize & Normalize
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    
    prediction = cnn_model.predict(img)
    return "Fake" if prediction > 0.5 else "Real"

print(predict_image("Dataset/Validation/Fake/fake_9.jpg"))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
Fake


In [98]:


def predict_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return "Error: Invalid image!"
    
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0  # Resize & Normalize
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    
    prediction = hybrid_model.predict(img)
    return "Fake" if prediction > 0.5 else "Real"

print(predict_image("Dataset/Test/Real/real_9.jpg"))



1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Fake


In [99]:


def predict_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return "Error: Invalid image!"
    
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0  # Resize & Normalize
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    
    prediction = hybrid_model.predict(img)
    return "Fake" if prediction > 0.5 else "Real"

print(predict_image("Dataset/Validation/Fake/fake_9.jpg"))



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
Real


In [100]:
hybrid_model.save("quantum_forgery_detector.h5")

In [ ]:
pip install fastapi uvicorn tensorflow pennylane pillow numpy

In [ ]:
pip install python-multipart